In [ ]:
from pathlib import Path
import json

import h5py
import numpy as np
import pandas as pd

from microns20.config import find_project_root, load_config

project_root = find_project_root()
config = load_config(project_root)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 180)

project_root

In [ ]:
# A Parquet viewer

def view_parquet(relative_path, rows=20):
    path = project_root / relative_path

    if not path.is_file():
        raise FileNotFoundError(path)

    dataframe = pd.read_parquet(path)

    print(f"File: {relative_path}")
    print(f"Shape: {dataframe.shape}")
    print(f"Columns: {len(dataframe.columns)}")

    display(dataframe.head(rows))

    return dataframe

In [ ]:
# Load all Stage 00–10 tables

tables = {
    "final20": "data/processed/final20_manifest.parquet",
    "functional_mappings":
        "data/processed/final20_functional_mappings.parquet",

    "recording_ranking":
        "results/tables/recording_selection_summary.parquet",

    "final_connectivity":
        "results/tables/final20_selection_connectivity.parquet",

    "morphology_manifest":
        "data/processed/morphologies/manifest.parquet",

    "morphology_qc":
        "results/tables/simulation_morphology_qc.parquet",

    "compartment_transitions":
        "results/tables/compartment_transition_events.parquet",

    "radius_normalization":
        "results/tables/radius_normalization_events.parquet",

    "intrinsic_synapses":
        "data/processed/connectivity/intrinsic_synapses.parquet",

    "external_incoming":
        "data/processed/connectivity/incoming_external_synapses.parquet",

    "external_sources":
        "data/processed/connectivity/external_presynaptic_nodes.parquet",

    "external_outgoing":
        "data/processed/connectivity/outgoing_external_synapses.parquet",

    "unresolved_synapses":
        "data/processed/connectivity/unresolved_synapses.parquet",

    "synapse_mapping_qc":
        "results/tables/synapse_mapping_qc.parquet",

    "sonata_build_qc":
        "results/tables/structural_sonata_build_qc.parquet",

    "end_to_end_validation":
        "results/tables/end_to_end_structural_validation.parquet",
}

loaded = {}

for name, relative_path in tables.items():
    dataframe = pd.read_parquet(project_root / relative_path)
    loaded[name] = dataframe

    print(
        f"{name:25s}"
        f" rows={len(dataframe):>7,}"
        f" columns={len(dataframe.columns):>3}"
    )

In [ ]:
# Build one table for our 20 neurons

manifest = loaded["final20"].copy()
functional = loaded["functional_mappings"].copy()
morphology_qc = loaded["morphology_qc"].copy()
synapse_qc = loaded["synapse_mapping_qc"].copy()

functional_summary = (
    functional
    .groupby("model_node_id")
    .agg(
        n_functional_units=("unit_id", "size"),
        functional_unit_ids=("unit_id", list),
        functional_fields=("field", list),
    )
    .reset_index()
)

population_overview = (
    manifest
    .merge(
        functional_summary,
        on="model_node_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        morphology_qc[
            [
                "model_node_id",
                "n_skeleton_points",
                "n_soma_points",
                "n_axon_points",
                "n_dendrite_points",
                "n_apical_points",
                "n_radius_normalized_points",
            ]
        ],
        on="model_node_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        synapse_qc,
        on=[
            "model_node_id",
            "nucleus_id",
            "pt_root_id",
        ],
        how="left",
        validate="one_to_one",
    )
)

display(population_overview)

# Save the population overview table .csv
population_overview.to_csv(
    project_root
    / "results/tables/final20_population_overview.csv",
    index=False,
)


In [ ]:
# Verify recording selection

recording_ranking = loaded["recording_ranking"].copy()

display(
    recording_ranking.sort_values("recording_rank")
)

selected_recording = recording_ranking.loc[
    recording_ranking["selected_recording"]
]

assert len(selected_recording) == 1

selected_recording

assert manifest["session"].nunique() == 1
assert manifest["scan_idx"].nunique() == 1

assert (
    manifest["session"].iloc[0]
    == selected_recording["session"].iloc[0]
)

assert (
    manifest["scan_idx"].iloc[0]
    == selected_recording["scan_idx"].iloc[0]
)

In [ ]:
# Inspect morphology

morphology_qc = loaded["morphology_qc"]

display(
    morphology_qc[
        [
            "model_node_id",
            "nucleus_id",
            "n_skeleton_points",
            "n_soma_points",
            "n_axon_points",
            "n_dendrite_points",
            "n_apical_points",
            "n_radius_normalized_points",
            "simulation_morphology_valid",
        ]
    ]
)

# inspect the actual radius corrections

display(
    loaded["radius_normalization"]
)

# inspect compartment transitions

transitions = loaded["compartment_transitions"]

display(transitions)

# filter out soma

non_soma_transitions = transitions.loc[
    transitions["parent_type"].ne(1)
    & transitions["child_type"].ne(1)
]

display(non_soma_transitions)

In [ ]:
# Inspect the connectivity

intrinsic = loaded["intrinsic_synapses"]

pair_summary = (
    intrinsic
    .groupby(
        ["pre_pt_root_id", "post_pt_root_id"],
        as_index=False,
    )
    .agg(
        n_synapses=("id", "size"),
    )
    .sort_values(
        "n_synapses",
        ascending=False,
    )
)

print("Individual recurrent contacts:", len(intrinsic))
print("Directed recurrent pairs:", len(pair_summary))

display(pair_summary)

In [ ]:
# Inspect external input

external = loaded["external_incoming"]

print(
    "External-to-selected20 contacts:",
    len(external),
)

print(
    "Unique external source roots:",
    external["pre_pt_root_id"].nunique(),
)


external_by_target = (
    external
    .groupby("model_node_id")
    .agg(
        n_external_contacts=("id", "size"),
        n_external_source_roots=(
            "pre_pt_root_id",
            "nunique",
        ),
    )
    .reset_index()
)

display(external_by_target)

In [ ]:
# Synapse placement QC

synapse_qc = loaded["synapse_mapping_qc"]

display(
    synapse_qc.sort_values(
        "placement_distance_max_um",
        ascending=False,
    )
)

# inspect the worst individual placements

all_inputs = pd.concat(
    [
        loaded["intrinsic_synapses"],
        loaded["external_incoming"],
    ],
    ignore_index=True,
)

worst_placements = (
    all_inputs
    .sort_values(
        "synapse_to_section_distance_um",
        ascending=False,
    )
    .head(50)
)

display(
    worst_placements[
        [
            "id",
            "model_node_id",
            "target_nucleus_id",
            "post_pt_root_id",
            "post_pt_supervoxel_id",
            "afferent_section_id",
            "afferent_section_pos",
            "afferent_section_type",
            "synapse_to_section_distance_um",
            "synapse_to_cave_skeleton_vertex_distance_um",
        ]
    ]
)

In [ ]:
# Target compartments

compartment_names = {
    1: "soma",
    2: "axon",
    3: "dendrite",
    4: "apical",
}

all_inputs["target_compartment"] = (
    all_inputs["afferent_section_type"]
    .map(compartment_names)
)

target_compartments = (
    all_inputs["target_compartment"]
    .value_counts()
    .rename_axis("compartment")
    .reset_index(name="n_synapses")
)

display(target_compartments)

In [ ]:
# Inspect SONATA files

def describe_h5(relative_path):
    path = project_root / relative_path

    print(f"\n{relative_path}")

    with h5py.File(path, "r") as handle:

        def visitor(name, obj):
            if isinstance(obj, h5py.Dataset):
                print(
                    f"{name:70s}",
                    obj.shape,
                    obj.dtype,
                )

        handle.visititems(visitor)
        
# call the function for each SONATA file
describe_h5(
    "data/processed/sonata/network/nodes/"
    "microns20_nodes.h5"
)

describe_h5(
    "data/processed/sonata/network/edges/"
    "microns20_to_microns20_edges.h5"
)


In [ ]:
# Explicit SONATA audit

selected_node_types = pd.read_csv(
    project_root
    / "data/processed/sonata/network/nodes/"
      "microns20_node_types.csv"
)

display(selected_node_types)

#

strict_checks = {
    "node_types_has_population":
        "population"
        in selected_node_types.columns,

    "node_types_has_recenter":
        "recenter"
        in selected_node_types.columns,
}

strict_checks